# 🧠 Retrieval-Augmented Generation (RAG) with LangChain, Groq & OpenAI
This notebook demonstrates a simple yet effective RAG pipeline using the LangChain framework.
It combines retrieval from a vector store with the power of LLMs for enhanced question answering.

Technologies used:
- `LangChain`
- `Groq` (DeepSeek LLM)
- `OpenAI` Embeddings
- `FAISS` Vector Store
- `.env` configuration

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

In [ ]:
llm=ChatGroq(model="deepseek-r1-distill-llama-70b")

In [ ]:
print(llm.invoke("What is the capital of France?").content)

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
from langchain.embeddings import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=api_key
)


In [ ]:
doc_vector=embedding_model.embed_query("What is the capital of France?")

In [ ]:
doc_vector

## Data Ingestion

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
os.getcwd()

In [ ]:
file_path=os.path.join(os.getcwd(), "data", "sample.pdf")

In [ ]:
loader=PyPDFLoader(file_path)

In [ ]:
documents=loader.load()

In [ ]:
documents

In [ ]:
len(documents)

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    length_function=len
)

In [ ]:
docs=text_splitter.split_documents(documents)

In [ ]:
len(docs)

In [ ]:
docs[0]

In [ ]:
docs[0].metadata

In [ ]:
docs[1].metadata

In [ ]:
docs[100].metadata

In [ ]:
docs[11].page_content

In [ ]:
from langchain.vectorstores import FAISS

In [ ]:
docs[0].page_content

In [ ]:
doc_vector=embedding_model.embed_documents(docs[0].page_content)

In [ ]:
len(embedding_model.embed_documents(docs[0].page_content)[0])

In [ ]:
vectorstore=FAISS.from_documents(docs, embedding_model)

## Retrieval-Augmented Generation (RAG)

In [ ]:
relevant_doc=vectorstore.similarity_search("where does tansfromeres come from?")

In [ ]:
relevant_doc

In [ ]:
relevant_doc[0].page_content

In [ ]:
relevant_doc=vectorstore.similarity_search("llama2 finetuning benchmark experiments.",k=10)

In [ ]:
relevant_doc

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [ ]:
retriever.invoke("llama2 finetuning benchmark experiments.")

### Question: user question
### Context: Top-k relevant chunks retrieved from the vector store using semantic search.


In [ ]:
prompt_template = """
        Answer the question based on the context provided below. 
        If the context does not contain sufficient information, respond with: 
        "I do not have enough information about this."

        Context: {context}

        Question: {question}

        Answer:"""

In [ ]:
from langchain.prompts import PromptTemplate

In [ ]:
prompt=PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [ ]:
prompt

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
parser=StrOutputParser()

In [ ]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [ ]:
from langchain_core.runnables import RunnablePassthrough

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("tell  me about the finetuning benchmark experiments?")